<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Build a simple page-level feature matrix for refresh ranking.
Only decision-time signals. No client names or URLs.

In [5]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

# ensure repo root
if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if not Path("/content/FlyRank-ML").exists():
        !git clone https://github.com/Fatima-05/FlyRank-ML.git /content/FlyRank-ML
    os.chdir("/content/FlyRank-ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df))

feature_cols = [c for c in [
    "impressions_90d", "clicks_90d", "ctr", "content_age_days",
    "word_count", "avg_position", "search_volume"
] if c in df.columns]

X = df[feature_cols].copy()
# simple fills
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

print("Feature columns:", feature_cols)
print(X.describe().T[["count", "mean", "min", "max"]])
X.head(3)

Rows: 30000
Feature columns: ['impressions_90d', 'clicks_90d', 'ctr', 'content_age_days', 'word_count', 'avg_position', 'search_volume']
                    count         mean   min       max
impressions_90d   30000.0  5200.366300   1.0  517715.0
clicks_90d        30000.0    16.097333   0.0    4178.0
ctr               30000.0     0.510733   0.0     100.0
content_age_days  30000.0   256.167800  90.0     564.0
word_count        30000.0  2310.205433   0.0    9546.0
avg_position      30000.0    16.342380   0.0     245.0
search_volume     30000.0   145.811667   0.0   74000.0


,impressions_90d,clicks_90d,ctr,content_age_days,word_count,avg_position,search_volume
0,3803,29,0.76,187,3221.0,10.6,10.0
1,15320,7,0.05,445,2481.0,20.3,90.0
2,12581,11,0.09,141,3515.0,36.5,0.0


## 2. Feature notes

For each feature:
- impressions_90d: demand/visibility proxy; missing → 0; known at decision time
- clicks_90d: engagement volume; missing → 0; known at decision time
- ctr: clicks/impressions style rate; missing → 0; known at decision time
- content_age_days: staleness proxy; missing → 0; known at decision time
- word_count: thin-content proxy; missing → 0; known at decision time
- avg_position: rank position if present; missing → 0; known at decision time
- search_volume: query demand proxy if present; missing → 0; known at decision time

None of these is a future outcome. None is a client name or URL.

In [6]:
notes = pd.DataFrame({
    "feature": feature_cols,
    "missing_fill": ["0"] * len(feature_cols),
    "available_before_predict": ["yes"] * len(feature_cols),
})
notes

,feature,missing_fill,available_before_predict
0,impressions_90d,0,yes
1,clicks_90d,0,yes
2,ctr,0,yes
3,content_age_days,0,yes
4,word_count,0,yes
5,avg_position,0,yes
6,search_volume,0,yes


## 3. The leakage hunt

Attack the feature set for:
1) label-derived columns
2) future windows
3) product flags that should not train the ranker

Show a simple test: label proxy must not appear in X.

In [7]:
df = df.copy()
df["is_declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

forbidden = [
    "trend_direction", "trend_pct", "is_declining",
    "client_name", "domain", "url", "query"
]
present_forbidden = [c for c in forbidden if c in X.columns]

print("Forbidden columns found inside X?", present_forbidden)
print("Label mean:", round(df["is_declining"].mean(), 3))

# trap demo: if we illegally add the label into features, score becomes unrealistically easy
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y = df["is_declining"]
X_clean = X.copy()
X_leak = X_clean.copy()
X_leak["is_declining_leaked"] = y.values

Xc_tr, Xc_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.25, random_state=42)
Xl_tr, Xl_te, _, _ = train_test_split(X_leak, y, test_size=0.25, random_state=42)

m_clean = LogisticRegression(max_iter=500).fit(Xc_tr, y_tr)
m_leak = LogisticRegression(max_iter=500).fit(Xl_tr, y_tr)

auc_clean = roc_auc_score(y_te, m_clean.predict_proba(Xc_te)[:, 1])
auc_leak = roc_auc_score(y_te, m_leak.predict_proba(Xl_te)[:, 1])

print(f"AUC clean features: {auc_clean:.3f}")
print(f"AUC with leaked label column: {auc_leak:.3f}")
print("If leaked AUC jumps near 1.0, that demonstrates the trap.")

Forbidden columns found inside X? []
Label mean: 0.542
AUC clean features: 0.592
AUC with leaked label column: 1.000
If leaked AUC jumps near 1.0, that demonstrates the trap.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. What I excluded and why

- trend_direction: this is the proxy label source; using it as a feature is leakage
- trend_pct (if present): label-like movement field
- client_id / content_id as features: identifiers, not generalizable signals (IDs may be used only for grouping/splits)
- client names, domains, URLs, queries: privacy / public-safety exclusions
- future-window outcomes: not knowable at decision time

In [8]:
excluded = pd.DataFrame({
    "field": [
        "trend_direction",
        "trend_pct",
        "client_id as feature",
        "content_id as feature",
        "client names / domains / URLs / queries",
        "future-window outcomes",
    ],
    "why": [
        "proxy-label source; leakage if used as feature",
        "label-like movement field",
        "identifier; grouping only",
        "identifier; display/join only",
        "privacy and public-safe rule",
        "not available at decision time",
    ],
})
excluded

,field,why
0,trend_direction,proxy-label source; leakage if used as feature
1,trend_pct,label-like movement field
2,client_id as feature,identifier; grouping only
3,content_id as feature,identifier; display/join only
4,client names / domains / URLs / queries,privacy and public-safe rule
5,future-window outcomes,not available at decision time


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.